In [1]:
# For text preprocessing
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

# For topic modeling
from gensim import corpora
from gensim.models import LdaModel
import pandas as pd

# Download NLTK Resources
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\nabie\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\nabie\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt.zip.
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\nabie\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [2]:
# Load the Data
documents = [
    "Rafael Nadal Joins Roger Federer in Missing U.S. Open",
    "Rafael Nadal Is Out of the Australian Open",
    "Biden Announces Virus Measures",
    "Biden's Virus Plans Meet Reality",
    "Where Biden's Virus Plan Stands"
]

In [4]:
# Initialize preprocessing tools
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    tokens = word_tokenize(text.lower()) # Tokenize and lowercase
    tokens = [token for token in tokens if token.isalnum()]  # Filter out non-alphanumeric tokens
    tokens = [token for token in tokens if token not in stop_words] # Remove stopwords
    tokens = [lemmatizer.lemmatize(token) for token in tokens] # Lemmatize each token
    return tokens

# Preprocess each document in the list
preprocessed_documents = [preprocess_text(doc) for doc in documents]
preprocessed_documents

[['rafael', 'nadal', 'join', 'roger', 'federer', 'missing', 'open'],
 ['rafael', 'nadal', 'australian', 'open'],
 ['biden', 'announces', 'virus', 'measure'],
 ['biden', 'virus', 'plan', 'meet', 'reality'],
 ['biden', 'virus', 'plan', 'stand']]

In [5]:
# Create a Gensim Dictionary object
dictionary = corpora.Dictionary(preprocessed_documents)

# Convert documents into bag-of-words representation
corpus = [dictionary.doc2bow(doc) for doc in preprocessed_documents]

In [6]:
# Train an LDA model with 2 topics
lda_model = LdaModel(corpus, num_topics=2, id2word=dictionary, passes=15)

In [7]:
# Store dominant topic labels for each document
article_labels = []

for i, doc in enumerate(preprocessed_documents):
    # Convert document to bow representation
    bow = dictionary.doc2bow(doc)
    # Get topic probabilities
    topics = lda_model.get_document_topics(bow)
    # Determine topic with highest probability
    dominant_topic = max(topics, key=lambda x: x[1])[0]
    # append to the list
    article_labels.append(dominant_topic)

In [8]:
# Create and print DataFrame
df = pd.DataFrame({"Article": documents, "Topic": article_labels})
print("Table with Articles and Topic:")
print(df)

Table with Articles and Topic:
                                             Article  Topic
0  Rafael Nadal Joins Roger Federer in Missing U....      1
1         Rafael Nadal Is Out of the Australian Open      1
2                     Biden Announces Virus Measures      0
3                   Biden's Virus Plans Meet Reality      1
4                    Where Biden's Virus Plan Stands      1


In [9]:
# Print top terms for each topic with weights
print("Top Terms for Each Topic:")
for idx, topic in lda_model.print_topics():
    print(f"Topic {idx}:")
    terms = [term.strip() for term in topic.split("+")]
    for term in terms:
        weight, word = term.split("*")
        print(f"- {word.strip()} (weight: {weight.strip()})")
    print()

Top Terms for Each Topic:
Topic 0:
- "biden" (weight: 0.135)
- "virus" (weight: 0.135)
- "announces" (weight: 0.119)
- "measure" (weight: 0.119)
- "stand" (weight: 0.043)
- "plan" (weight: 0.042)
- "meet" (weight: 0.042)
- "reality" (weight: 0.042)
- "australian" (weight: 0.041)
- "nadal" (weight: 0.041)

Topic 1:
- "open" (weight: 0.091)
- "rafael" (weight: 0.091)
- "nadal" (weight: 0.091)
- "plan" (weight: 0.090)
- "virus" (weight: 0.084)
- "biden" (weight: 0.084)
- "federer" (weight: 0.054)
- "join" (weight: 0.054)
- "missing" (weight: 0.054)
- "roger" (weight: 0.054)

